# Email Summarizer Workflow

In this demo you will build a **Foundry Workflow** that accepts a raw email,
sends it to an AI agent for analysis, and displays a clean summary.

### Workflows vs Agents

| | **Agent** | **Workflow** |
|---|---|---|
| **What it is** | A single intelligent unit: model + instructions + tools | An orchestration layer that coordinates agents into a multi-step process |
| **How it's created** | `PromptAgentDefinition` | `WorkflowAgentDefinition` with CSDL YAML |
| **Shows up in** | Portal → Agents | Portal → Workflows |

In this notebook we:
1. **Register** an agent in Foundry using the Python SDK
2. **Define** a workflow in YAML that calls that agent
3. **Register** the workflow so it appears in the portal
4. **Test** the workflow executes end-to-end
5. **Invoke** the agent via the Conversations API to see its output

## Step 1: Install Packages

In [ ]:
%pip install python-dotenv azure-ai-projects==2.2.0 azure-identity

## Step 2: Configure Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)
load_dotenv(override=True)

project_url = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

print("Project URL:", project_url)
print("Deployment:", deployment_name)

## Step 3: Create the Foundry Client & Helpers

We use the **sync** `AIProjectClient` to register agents and workflows
in Foundry. The `get_openai_client()` method returns an OpenAI-compatible
client for talking to registered agents via the Conversations API.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WorkflowAgentDefinition
from azure.identity import DefaultAzureCredential

foundry_client = AIProjectClient(
    endpoint=project_url,
    credential=DefaultAzureCredential(),
    allow_preview=True,  # workflows are a preview feature; required to register WorkflowAgentDefinition
)

oai = foundry_client.get_openai_client()


def register_agent(name: str, instructions: str, description: str = ""):
    """Register a prompt agent in Foundry."""
    result = foundry_client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(
            model=deployment_name,
            instructions=instructions,
        ),
        description=description,
    )
    print(f"Registered agent: {result.name} (version {result.version})")
    return result


def register_workflow(name: str, yaml_str: str, description: str = ""):
    """Register a workflow in Foundry."""
    result = foundry_client.agents.create_version(
        agent_name=name,
        definition=WorkflowAgentDefinition(workflow=yaml_str),
        description=description,
    )
    print(f"Registered workflow: {result.name} (version {result.version})")
    return result


def invoke_agent(agent_name: str, user_input: str):
    """Create a conversation and invoke a registered agent."""
    session = oai.conversations.create()
    response = oai.responses.create(
        conversation=session.id,
        input=user_input,
        extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
    )
    return response.output_text


print("Foundry client ready.")

## Step 4: Register the Email Summarizer Agent

This registers a **prompt agent** in Foundry with detailed email analysis
instructions. The agent will appear in the portal under **Agents**.

In [ ]:
SUMMARIZER_INSTRUCTIONS = """You are an expert email analyst. When the user provides the text of an email, you must produce a clear, well-organized summary using the exact format below.

Your summary MUST include all of the following sections in this order:

## Subject Line
Extract or infer the subject of the email. If no explicit subject line is present, write a concise one that captures the main topic.

## Sender's Intent
In 1-2 sentences, explain why the sender wrote this email. What are they trying to accomplish?

## Key Points
Provide a bullet list of the most important pieces of information in the email. Aim for 3-6 bullets. Be concise -- one sentence per bullet.

## Action Items
List every action item mentioned or implied in the email. For each action item include:
- What needs to be done
- Who is responsible (if stated)
- The deadline (if stated; otherwise write "No deadline specified")

If there are no action items, write "No action items identified."

## Urgency Level
Rate the urgency of this email as one of the following:
- **Low** -- informational only, no time pressure
- **Medium** -- requires attention within a few days
- **High** -- requires immediate attention or response

Provide a one-sentence justification for your rating.

---

Rules:
- Do NOT include any information that is not in the original email. Do not guess or fabricate details.
- Keep the summary shorter than the original email whenever possible.
- Use professional, neutral language regardless of the tone of the original email.
- If the email is very short (one or two sentences), still produce all five sections -- just keep each section brief."""

register_agent(
    name="email-summarizer",
    instructions=SUMMARIZER_INSTRUCTIONS,
    description="Analyzes emails and produces structured summaries with subject, intent, key points, action items, and urgency.",
)

## Step 5: Define and Register the Workflow

The workflow is defined in **CSDL YAML** -- the declarative language used by
Foundry Workflows. This simple workflow:
1. Captures the user's message via `SetVariable`
2. Sends it to the `email-summarizer` agent via `InvokeAzureAgent`
3. Ends the conversation

Once registered, the workflow will appear in the portal under **Workflows**
where you can visualize it, edit it, and test it interactively.

In [ ]:
WORKFLOW_YAML = """
kind: workflow
name: email-summarizer-workflow
description: Accepts a raw email and returns a structured summary via the Email Summarizer agent.
trigger:
  kind: OnConversationStart
  id: trigger_start
  actions:
    - kind: SetVariable
      id: init_msg
      variable: Local.LatestMessage
      value: =UserMessage(System.LastMessageText)

    - kind: InvokeAzureAgent
      id: invoke_summarizer
      agent:
        name: email-summarizer
        conversationId: =System.ConversationId
      input:
        messages: =Local.LatestMessage
      output:
        messages: Local.LatestMessage
      autoSend: true

    - kind: EndConversation
      id: end_conversation
""".strip()

register_workflow(
    name="email-summarizer-workflow",
    yaml_str=WORKFLOW_YAML,
    description="Simple workflow: email in, structured summary out.",
)

## Step 6: Verify the Workflow Executes

Let's confirm the workflow runs end-to-end by invoking it via the Conversations
API. We check that all three actions (SetVariable → InvokeAzureAgent →
EndConversation) complete successfully.

> **Note:** In the current SDK preview, the workflow's agent response is
> delivered inline during execution (`autoSend`) but is not included in the
> Python SDK's `output_text` field. To see the full rendered response, use
> the portal's built-in test pane (Step 8) or invoke the agent directly (Step 7).

In [ ]:
# Run the workflow and inspect the execution trace
session = oai.conversations.create()
response = oai.responses.create(
    conversation=session.id,
    input="From: Test <test@example.com>\nSubject: Hello\n\nJust a quick test.",
    extra_body={"agent_reference": {"name": "email-summarizer-workflow", "type": "agent_reference"}},
)

print("Workflow execution trace:")
try:
    for action in response.output:
        print(f"  Action: {action.action_id} | Kind: {action.kind} | Status: {action.status}")
    print(f"\nAll {len(response.output)} actions completed successfully!")
except AttributeError:
    # Response format may vary — print raw output if trace attributes are not available
    print("Workflow response:")
    for item in response.output:
        print(f"  {item}")

## Step 7: Test the Agent

Now let's invoke the registered `email-summarizer` agent directly via the
Conversations API to see its output.

### Test Email: Short Meeting Request

In [ ]:
test_email_1 = """From: Lisa Chen <lisa.chen@acme.com>
To: You
Date: Monday, March 10, 2025 9:14 AM
Subject: Quick sync on Q2 planning

Hi,

Do you have 30 minutes this week to chat about the Q2 marketing budget? I'd like to align
on priorities before the leadership review on Friday.

I'm free Wednesday 2-4 PM or Thursday morning. Let me know what works.

Thanks,
Lisa"""

print(invoke_agent("email-summarizer", test_email_1))